# Main Pipeline

Runs the whole project from start to finish, in order:

1. Collect data (SAP News, GNews, Hacker News)
2. Clean the data
3. Build embeddings + FAISS index
4. Run the opportunity, risk, trend, and competitor agents
5. Run the CEO agent

After this notebook finishes, run `streamlit run app.py` to see the dashboard.

Before running this, make sure Ollama is installed and running:
```
bash setup.sh
ollama list
```


In [ ]:
import subprocess
import time

# nbconvert default timeout is 60 sec, way too short for embedding/LLM steps
TIMEOUT = 3600  # 1 hour per notebook - local CPU LLM calls can each take up to 15 min


def run_notebook(path):
    print("Running:", path)
    start = time.time()

    subprocess.run([
        "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--inplace",
        f"--ExecutePreprocessor.timeout={TIMEOUT}",
        path,
    ], check=True)

    minutes = (time.time() - start) / 60
    print(f"Done: {path}  ({minutes:.1f} min)")
    print()


## Step 1: Collect data

In [ ]:
collection_notebooks = [
    "01_Data_Collection_cleaned/01_sap_news.ipynb",
    "01_Data_Collection_cleaned/02_gnew.ipynb",
    "01_Data_Collection_cleaned/03_hackernews.ipynb",
]

for nb_path in collection_notebooks:
    run_notebook(nb_path)


## Step 2: Clean the data

In [ ]:
run_notebook("notebook/data_cleaning.ipynb")


## Step 3: Build embeddings + FAISS index

In [ ]:
run_notebook("notebook/embedding_v2.ipynb")


## Step 4: Run the strategic intelligence agents

In [ ]:
agent_notebooks = [
    "agents/01_opportunity_agent.ipynb",
    "agents/02_risk_agent.ipynb",
    "agents/03_trend_agent.ipynb",
    "agents/04_competitor_monitor.ipynb",
]

for nb_path in agent_notebooks:
    run_notebook(nb_path)


## Step 5: Run the CEO agent

In [ ]:
run_notebook("agents/05_ceo_agent.ipynb")


## Done

All the data files needed by the dashboard should now be in `notebook/data/`:
- `clean_data.json`
- `chunked_data.json`, `chunk_embeddings.npy`, `sap_intelligence.index`
- `opportunities.json`, `risks.json`, `trends.json`, `competitor_activity.json`
- `recommendations.json`, `ceo_briefing.json`

Run the dashboard with:
```
streamlit run app.py
```


In [ ]:
import os

data_dir = "notebook/data"
expected_files = [
    "clean_data.json",
    "chunked_data.json",
    "chunk_embeddings.npy",
    "sap_intelligence.index",
    "opportunities.json",
    "risks.json",
    "trends.json",
    "competitor_activity.json",
    "recommendations.json",
    "ceo_briefing.json",
]

print("Checking output files:")
for f in expected_files:
    path = os.path.join(data_dir, f)
    exists = os.path.exists(path)
    print(f"  [{'OK' if exists else 'MISSING'}] {f}")
